# Export and process the pinned DuYu/Hanzi-Pinyin corpus

Export one explicitly pinned DuYu split with its supplied labels. After the sound inventory is available, run the independent local processing section against the existing `corpus.csv` to publish an immutable processed-corpus bundle for final assembly.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'datasets').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Could not find a project root containing datasets/ and src/')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from phonetics.artifacts import sha256_file
from phonetics.source_preparation import export_duyu

DUYU_EXPORT_DIR = PROJECT_ROOT / "datasets/source-data/duyu-hanzi-pinyin"
DUYU_REVISION = "9961d7d05910be7006f2dcfae21f4fc5ce4d560a"
DUYU_SPLIT = "train"
# Historical columns verified for this pinned revision.
DUYU_COLUMNS = {"text": "我们试试看！", "pinyin": "wo3 men shi4 shi4 kan4 ！"}
TONE_CONVENTION = "neutral_unmarked"


In [2]:
assert "REPLACE_WITH" not in repr((DUYU_EXPORT_DIR, DUYU_REVISION, DUYU_SPLIT, DUYU_COLUMNS))
exported_path = export_duyu(DUYU_EXPORT_DIR, revision=DUYU_REVISION, split=DUYU_SPLIT,
                            columns=DUYU_COLUMNS, tone_convention=TONE_CONVENTION, allow_download=True)
print(exported_path)
print("corpus.csv sha256:", sha256_file(exported_path / "corpus.csv"))


d:\projects\Jupyter\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\projects\Mandarin-Phonetics-Modeling\project\datasets\source-data\duyu-hanzi-pinyin
corpus.csv sha256: 0a321b103c495c506f408223798cb90ab42a36d796428f361e9476c2665a7792


## Process the exported corpus locally

Run this section independently after `corpus.csv` already exists and after an acoustic bundle has supplied a validated sound inventory. This stage does not run inference and does not reload DuYu. It normalizes labels, rejects every row for texts with conflicting canonical pronunciations, applies Wubi and acoustic coverage filters, merges identical duplicates, and samples the survivors.


In [3]:
from phonetics import prepare_corpus

RUN_PROCESSING = True
RAW_CORPUS_PATH = DUYU_EXPORT_DIR / 'corpus.csv'
PROCESSED_CORPUS_OUTPUT_ROOT = PROJECT_ROOT / 'datasets/processed-corpus'
ACOUSTIC_BUNDLE_PATH = PROJECT_ROOT / 'datasets/pinyin-embeddings/duyu-chinese-lessons-zh-cn-v1'
ACOUSTIC_MANIFEST_SHA256 = '25aef7ef70ec8687dda32afa92887130cd815d835f7b4ff6b9344822eb412d11'

PROCESSING_CONFIG = {
    'corpus_id': 'duyu-chinese-lessons-v1',
    'output_root': str(PROCESSED_CORPUS_OUTPUT_ROOT),
    'corpus': {'path': str(RAW_CORPUS_PATH), 'sha256': sha256_file(RAW_CORPUS_PATH) if RUN_PROCESSING else None, 'source_id': 'duyu', 'tone_convention': 'explicit'},
    'wubi': {'path': str(PROJECT_ROOT / 'datasets/wubi/chinese_to_wubi.pkl'), 'sha256': '80d3228546097490632cfdc1b42666b706840e2d7f1f76f64712449c9cfe927e'},
    'sampling': {'size': 100_000, 'strategy': 'uniform', 'seed': 42},
    'acoustic_bundle': {'path': str(ACOUSTIC_BUNDLE_PATH), 'manifest_sha256': ACOUSTIC_MANIFEST_SHA256},
}


In [4]:
if RUN_PROCESSING:
    assert 'REPLACE_WITH' not in repr(PROCESSING_CONFIG)
    processed_corpus_path = prepare_corpus(PROCESSING_CONFIG)
    print(processed_corpus_path)
    print('manifest.json sha256:', sha256_file(processed_corpus_path / 'manifest.json'))
else:
    print('Corpus processing disabled; run this section after the raw export and sound inventory are available.')


D:\projects\Mandarin-Phonetics-Modeling\project\datasets\processed-corpus\duyu-chinese-lessons-v1
manifest.json sha256: 4727c1f2e8d607c0d8dd4ef79a07c7e1764e1ea76bbcfaeafa66b2aaff5ceb5f
